# SpaceX Falcon 9 landing analysis

Reproducible study using IBM teaching snapshots and archived Wikipedia records.

AI-assisted project draft for learner review. Read the source code and verify your course rules before submission. The community API was unavailable during this run; no live-API collection result is claimed. Dataset cohorts and outcome definitions differ.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import analyze
from dashboard import figures
from build_map import build_map
ROOT = Path.cwd()
sns.set_theme(style="whitegrid")

## 1. Sources and provenance
Downloaded snapshots retain original bytes and SHA-256 hashes in data/sources.json. The optional collect_api.py refresh script is provided separately.

In [ ]:
display(pd.DataFrame(json.loads((ROOT/"data/sources.json").read_text()))[["file","status","url"]])

## 2. Wrangling and outcome definitions
Course Class=1 means Outcome starts with True, including five controlled ocean landings. Other outcomes include no attempt. Class therefore approximates landing success and does not directly measure economically reusable recovery. The IBM snapshot already contains mean-imputed payload values.

In [ ]:
d=analyze.wrangle()
display(d.head())
display(d.Outcome.value_counts())
display(d.isna().sum().rename("missing"))

## 3. Web scraping and SQL
BeautifulSoup selects launch tables, removes footnote tags, excludes narrative rows, and parses dates and kilogram payloads. SQL uses this separate scraped cohort, not the 90-row ML dataset. Raw site labels are retained and normalized into SiteGroup for comparison.

In [ ]:
scraped,sql=analyze.sql_analysis()
print(f"{len(scraped)} unique numbered Falcon 9 launches: {scraped.Date.min()} to {scraped.Date.max()}")
display(scraped.head())

In [ ]:
for name,result in sql.items():
    print(name)
    print(analyze.SQL[name])
    display(pd.DataFrame(result))

## 4. Exploratory data analysis
Color indicates the course landing Class, not orbital mission outcome. These are associations in a small historical cohort, not causal effects.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(data=d,x="FlightNumber",y="LaunchSite",hue="Class",ax=axes[0])
sns.scatterplot(data=d,x="PayloadMass",y="LaunchSite",hue="Class",ax=axes[1])
axes[0].set_title("Flight number and launch site")
axes[1].set_title("Payload (kg) and launch site")
plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
d.groupby("Orbit").Class.mean().plot.bar(ax=axes[0],color="#087f8c")
d.groupby(d.Date.dt.year).Class.mean().plot(ax=axes[1],marker="o",color="#087f8c")
axes[0].set(title="Success fraction by orbit",ylabel="Course Class=1 fraction",ylim=(0,1.05))
axes[1].set(title="Yearly success trend",ylabel="Course Class=1 fraction",ylim=(0,1.05))
plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(data=d,x="FlightNumber",y="Orbit",hue="Class",ax=axes[0])
sns.scatterplot(data=d,x="PayloadMass",y="Orbit",hue="Class",ax=axes[1])
plt.tight_layout();plt.show()

## 5. Interactive analytics
The Folium HTML files contain markers, outcome records and pad-to-pad proximity. The notebook outputs below come directly from the Dash callback function. Run python dashboard.py to use the site dropdown and payload slider. The separate 56-row dashboard and geographical snapshots have older coverage.

In [ ]:
geo=build_map()
display(pd.DataFrame(geo["sites"]))
print(f"Distance between supplied CCAFS LC-40 and KSC LC-39A coordinates: {geo["cape_to_ksc_km"]:.2f} km")
print("Open results/launch_sites.html, launch_records.html, and proximity.html in a browser.")

In [ ]:
pie,scatter,count=figures("ALL",[0,10000])
print(count)
display(pie)
display(scatter)

In [ ]:
pie,scatter,count=figures("KSC LC-39A",[0,10000])
print(count)
display(pie)
display(scatter)

## 6. Predictive analysis
Split 72/18, stratified, random_state=42. Fit imputation, scaling and one-hot encoding inside each 5-fold training CV split. Choose the model by training CV accuracy, then evaluate once on the holdout. Outcome, Class, Serial, LandingPad and lifetime ReusedCount never enter features. A separate chronological split repeats training-only model selection.

In [ ]:
model=analyze.modeling(d)
display(pd.DataFrame(model["scores"]))
print("Chosen by CV:",model["best_model"])
print("Majority baseline:",model["baseline_accuracy"])

In [ ]:
cm=model["confusion_matrix"]
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Other","Success"],yticklabels=["Other","Success"])
plt.xlabel("Predicted");plt.ylabel("Actual");plt.title("SVM: untouched 18-row holdout");plt.show()
print("Chronological model:",model["chronological_model"])
print("Chronological accuracy:",model["chronological_accuracy"])
print("Chronological majority baseline:",model["chronological_test_successes"]/18)

## 7. Interpretation and limits
The model is educational, not a launch-safety or pricing system. An 18-row test set has high uncertainty. Site, mission era and payload are confounded. The Class=1 label includes ocean landings and does not measure refurbishment cost. Historical performance should not be extrapolated to current launches. A future extension should use raw missing payloads, mission-time core histories, grouped or rolling validation, probability calibration and actual recovery economics.